In [ ]:
import torch
import torchvision
import json
import os
from tqdm import tqdm
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torchvision.transforms import functional as F
from ultralytics import YOLO

VAL_IMG_DIR = "../data/valid/images"
VAL_ANN_FILE = "../data/annotations_val.json"
DATA_YAML = "../data/data.yaml"
CONF_THRESH = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = {}

def coco_eval(preds, coco_gt):
    output_json = "../results/temp_eval.json"
    with open(output_json, 'w') as f:
        json.dump(preds, f)
    coco_dt = coco_gt.loadRes(output_json)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    import numpy as np
    ap50 = float(coco_eval.stats[1])
    ap = float(coco_eval.stats[0])
    recall = float(coco_eval.stats[8])
    prec_vals = coco_eval.eval['precision'][0, :, 0, 0, 2]
    valid_prec = prec_vals[prec_vals >= 0]
    precision = float(np.mean(valid_prec)) if len(valid_prec) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
    return {"mAP@0.5": ap50, "mAP@0.5:0.95": ap, "Precision": precision, "Recall": recall, "F1": f1}


def eval_yolo(model_path):
    model = YOLO(model_path)
    metrics = model.val(data=DATA_YAML, imgsz=640, device=DEVICE.index if DEVICE.type == "cuda" else "cpu")
    return {
        "mAP@0.5": metrics.box.map50,
        "mAP@0.5:0.95": metrics.box.map,
        "Precision": metrics.box.p[0],
        "Recall": metrics.box.r[0],
        "F1": metrics.box.f1[0]
    }

def eval_ssd(model_path):
    from torchvision.models.detection import ssd300_vgg16
    model = ssd300_vgg16(weights="DEFAULT")
    in_channels = [512, 1024, 512, 256, 256, 256]
    num_anchors = [4, 6, 6, 6, 4, 4]
    model.head.classification_head = torchvision.models.detection.ssd.SSDClassificationHead(
        in_channels=in_channels, num_anchors=num_anchors, num_classes=2
    )
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE).eval()

    coco = COCO(VAL_ANN_FILE)
    preds = []
    for img_info in tqdm(coco.loadImgs(coco.getImgIds()), desc="SSD Eval"):
        path = os.path.join(VAL_IMG_DIR, img_info['file_name'])
        img = Image.open(path).convert("RGB")
        tensor = F.to_tensor(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(tensor)[0]

        for box, score, label in zip(output['boxes'], output['scores'], output['labels']):
            if score < CONF_THRESH:
                continue
            x1, y1, x2, y2 = box.cpu().numpy()
            preds.append({
                "image_id": img_info['id'],
                "category_id": int(label),
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                "score": float(score)
            })

    return coco_eval(preds, coco)

def eval_frcnn(model_path):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, 2)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE).eval()

    coco = COCO(VAL_ANN_FILE)
    preds = []
    for img_info in tqdm(coco.loadImgs(coco.getImgIds()), desc="Faster R-CNN Eval"):
        path = os.path.join(VAL_IMG_DIR, img_info['file_name'])
        img = Image.open(path).convert("RGB")
        tensor = F.to_tensor(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(tensor)[0]

        for box, score, label in zip(output['boxes'], output['scores'], output['labels']):
            if score < CONF_THRESH:
                continue
            x1, y1, x2, y2 = box.cpu().numpy()
            preds.append({
                "image_id": img_info['id'],
                "category_id": int(label),
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                "score": float(score)
            })

    return coco_eval(preds, coco)

print("🧪 Evaluating YOLO...")
results["YOLO"] = eval_yolo("../models/yolov11/tuned.pt")

print("\n🧪 Evaluating SSD...")
results["SSD"] = eval_ssd("../models/ssd/ssd_speedbump.pth")

print("\n🧪 Evaluating Faster R-CNN...")
results["Faster R-CNN"] = eval_frcnn("../models/faster_rcnn/fasterrcnn_speedbump.pth")

print("\n📊 Model Comparison:")
header = ["Model", "mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1 Score", "Accuracy"]
print("{:<15} {:<10} {:<14} {:<10} {:<10} {:<10} {:<10}".format(*header))
for model, metrics in results.items():
    precision = metrics["Precision"]
    recall = metrics["Recall"]
    f1 = metrics["F1"]
    accuracy = (precision + recall) / 2
    print("{:<15} {:<10.4f} {:<14.4f} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f}".format(
        model,
        metrics["mAP@0.5"],
        metrics["mAP@0.5:0.95"],
        precision,
        recall,
        f1,
        accuracy
    ))



🧪 Evaluating YOLO...
Ultralytics 8.3.145  Python-3.12.4 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16310MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 199.192.2 MB/s, size: 89.7 KB)


val: Scanning C:\Users\faizj\OneDrive\Desktop\others\SIKE\FYP\Model\Model Yolov11\valid\labels.cache... 898 images, 0 backgrounds, 0 corrupt: 100%|██████████| 898/898 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 57/57 [00:09<00:00,  5.93it/s]


                   all        898       1038      0.923      0.824      0.895       0.64
Speed: 0.2ms preprocess, 2.2ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to runs\detect\val3

🧪 Evaluating SSD...
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


SSD Eval: 100%|██████████| 898/898 [00:33<00:00, 26.81it/s]


Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.20s).
Accumulating evaluation results...
DONE (t=0.06s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.394
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.635
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.407
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.189
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.453
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.430
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.435
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.435
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10

Faster R-CNN Eval: 100%|██████████| 898/898 [01:05<00:00, 13.68it/s]


Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.20s).
Accumulating evaluation results...
DONE (t=0.04s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.520
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.820
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.571
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.013
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.405
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.559
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.552
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.580
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.580
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10

In [ ]:
import torch
import torchvision
import json
import os
from tqdm import tqdm
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torchvision.transforms import functional as F
from ultralytics import YOLO
import time  # For speed measurement

VAL_IMG_DIR = "../data/valid/images"
VAL_ANN_FILE = "../data/annotations_val.json"
DATA_YAML = "../data/data.yaml"
CONF_THRESH = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def coco_eval(preds, coco_gt):
    output_json = "../results/temp_eval.json"
    with open(output_json, 'w') as f:
        json.dump(preds, f)
    coco_dt = coco_gt.loadRes(output_json)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    import numpy as np
    prec_vals = coco_eval.eval['precision'][0, :, 0, 0, 2]
    valid_prec = prec_vals[prec_vals >= 0]
    precision = float(np.mean(valid_prec)) if len(valid_prec) > 0 else 0.0
    recall = float(coco_eval.stats[8])
    return {
        "mAP@0.5": float(coco_eval.stats[1]),
        "Precision": precision,
        "Recall": recall,
        "F1": 2 * (precision * recall) / (precision + recall + 1e-6),
        "Inference_time": None
    }

def eval_yolo(model_path):
    model = YOLO(model_path)
    
    # First run for warmup
    _ = model.predict("../data/valid/images/7.jpg", imgsz=640, conf=CONF_THRESH)
    
    # Speed test
    start_time = time.time()
    metrics = model.val(data=DATA_YAML, imgsz=640, device=DEVICE.type, conf=CONF_THRESH)
    inference_time = (time.time() - start_time) / len(metrics.speed) if hasattr(metrics, 'speed') else None
    
    return {
        "mAP@0.5": metrics.box.map50,
        "Precision": metrics.box.p,
        "Recall": metrics.box.r,
        "F1": metrics.box.f1,
        "Inference_time": inference_time
    }

def eval_torchvision_model(model, model_name):
    coco = COCO(VAL_ANN_FILE)
    preds = []
    total_time = 0
    count = 0
    
    for img_info in tqdm(coco.loadImgs(coco.getImgIds()), desc=f"{model_name} Eval"):
        path = os.path.join(VAL_IMG_DIR, img_info['file_name'])
        try:
            img = Image.open(path).convert("RGB")
            tensor = F.to_tensor(img).unsqueeze(0).to(DEVICE)

            start_time = time.time()
            with torch.no_grad():
                output = model(tensor)[0]
            total_time += time.time() - start_time
            count += 1

            for box, score, label in zip(output['boxes'], output['scores'], output['labels']):
                if score < CONF_THRESH:
                    continue
                x1, y1, x2, y2 = box.cpu().numpy()
                preds.append({
                    "image_id": img_info['id'],
                    "category_id": int(label),
                    "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                    "score": float(score)
                })
        except Exception as e:
            print(f"Error processing {path}: {str(e)}")
    
    metrics = coco_eval(preds, coco)
    metrics["Inference_time"] = total_time / count if count > 0 else None
    return metrics

def print_results(results):
    print("\n🚗 Speed Bump Detection Model Comparison")
    print("="*70)
    print("{:<15} {:<10} {:<10} {:<10} {:<10} {:<15}".format(
        "Model", "mAP@0.5", "Precision", "Recall", "F1", "Inference Time (ms)"))
    print("-"*70)
    
    for model, metrics in results.items():
        # Extract first element if metric is an array
        map50 = metrics["mAP@0.5"].item() if hasattr(metrics["mAP@0.5"], 'item') else metrics["mAP@0.5"]
        precision = metrics["Precision"].item() if hasattr(metrics["Precision"], 'item') else metrics["Precision"]
        recall = metrics["Recall"].item() if hasattr(metrics["Recall"], 'item') else metrics["Recall"]
        f1 = metrics["F1"].item() if hasattr(metrics["F1"], 'item') else metrics["F1"]
        inference_time = metrics["Inference_time"] * 1000 if metrics["Inference_time"] else -1
        
        print("{:<15} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f} {:<15.2f}".format(
            model,
            float(map50),
            float(precision),
            float(recall),
            float(f1),
            float(inference_time)
        ))

# Main evaluation
results = {}

print("🧪 Evaluating YOLO...")
results["YOLO"] = eval_yolo("../models/yolov11/tuned.pt")

print("\n🧪 Evaluating SSD...")
ssd_model = torchvision.models.detection.ssd300_vgg16(weights="DEFAULT")
ssd_model.head.classification_head = torchvision.models.detection.ssd.SSDClassificationHead(
    in_channels=[512, 1024, 512, 256, 256, 256],
    num_anchors=[4, 6, 6, 6, 4, 4],
    num_classes=2
)
ssd_model.load_state_dict(torch.load("../models/ssd/ssd_speedbump.pth", map_location=DEVICE))
ssd_model.to(DEVICE).eval()
results["SSD"] = eval_torchvision_model(ssd_model, "SSD")

print("\n🧪 Evaluating Faster R-CNN...")
frcnn_model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = frcnn_model.roi_heads.box_predictor.cls_score.in_features
frcnn_model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, 2)
frcnn_model.load_state_dict(torch.load("../models/faster_rcnn/fasterrcnn_speedbump.pth", map_location=DEVICE))
frcnn_model.to(DEVICE).eval()
results["Faster R-CNN"] = eval_torchvision_model(frcnn_model, "Faster R-CNN")

print_results(results)

🧪 Evaluating YOLO...

image 1/1 c:\Users\faizj\OneDrive\Desktop\others\SIKE\FYP\Model\dataset\val\images\7.jpg: 640x384 1 speed_bump, 9.3ms
Speed: 3.5ms preprocess, 9.3ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 384)
Ultralytics 8.3.145  Python-3.12.4 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16310MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 1036.4544.6 MB/s, size: 326.0 KB)


val: Scanning C:\Users\faizj\OneDrive\Desktop\others\SIKE\FYP\Model\Model Yolov11\valid\labels.cache... 898 images, 0 backgrounds, 0 corrupt: 100%|██████████| 898/898 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 57/57 [00:04<00:00, 12.08it/s]


                   all        898       1038      0.923      0.823      0.893      0.683
Speed: 0.1ms preprocess, 1.6ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs\detect\val6

🧪 Evaluating SSD...


c:\Users\faizj\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\faizj\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=SSD300_VGG16_Weights.COCO_V1`. You can also use `weights=SSD300_VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


...

Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.16s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.394
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.635
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.407
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.189
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.453
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.430
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.435
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.435
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10

c:\Users\faizj\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


Faster R-CNN Eval: 100%|██████████| 898/898 [00:46<00:00, 19.28it/s]


Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.17s).
Accumulating evaluation results...
DONE (t=0.04s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.520
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.820
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.571
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.013
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.405
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.559
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.552
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.580
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.580
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10